# ICA14 - NLP Sentiment

Sentiment analysis using (some) NLP approaches we discussed in class.

In [ ]:
# If you don't have the data, you can get it from kagglehub

import kagglehub

path = kagglehub.dataset_download("ykhorrami/sentiment-labelled-sentences")


In [ ]:
# Read in all the files to create a unified dataset
import pandas as pd
import glob
import os

path = r'./' # use your path
all_files = glob.glob(os.path.join(path , "*.txt"))

li = []

for filename in all_files:
    frame = pd.read_csv(filename, index_col=None, header=None, sep='\t')
    li.append(frame)

df = pd.concat(li, axis=0, ignore_index=True)
df

In [ ]:
# Shuffle the data, resetting all the rows -- don't want the amazon stuff always on top
df = df.sample(frac=1).reset_index(drop=True)
df

In [ ]:
df.describe()

In [ ]:
# Train/test split -- 80/20?
from sklearn.model_selection import train_test_split

train_x, test_x, train_y, test_y = train_test_split(df.iloc[:,:-1], df.iloc[:,-1], test_size=0.2)

In [ ]:
print('Training data/labels:', train_x.shape, train_y.shape)
print('Testing data/labels:', test_x.shape, test_y.shape)

There is a pre-trained model called VADER. It's a simple model: looks at the text for positive / negative tokens. These get compounded into a score.

In [ ]:
# If you need to install NLTK, this is one way. (You can also do this on command line.)
!pip install nltk

In [ ]:
# VADER is a pretrained tool focused on sentiment analysis.

import nltk

nltk.download('vader_lexicon')

In [ ]:
# VADER example
from nltk.sentiment.vader import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

print('Sentence: ', train_x.iloc[0,0])
# Here's how you get the different possible scores -- a negative, neutral, positive and compound score
print('VADER scores: ', vader.polarity_scores(train_x.iloc[0,0]))

In [ ]:
vader.polarity_scores(train_x.iloc[0,0])['compound']

In [ ]:
# The compound is the only score we need. These scores range from -1 to +1
# What is the VADER hypothesis for the above sentence?

In [ ]:
# Get sentiment scores for all the training corpus
train_x.iloc[:,0].apply(lambda x: vader.polarity_scores(x)['compound'])

In [ ]:
# Our corpus' labels do not align with VADER's We'll need to squash the scores from VADER.
def squash(scores):
    # TODO: Implement this
    # * Input is a list of scores from VADER
    # * Output is a list of (converted) scores aligned to our corpus
    pass

In [ ]:
squash(test_x.iloc[:,0].apply(lambda x: vader.polarity_scores(x)['compound']))[:10]

In [ ]:
# Let's put this all together into a prediction system
from sklearn.metrics import accuracy_score

accuracy_score(test_y, squash(test_x.iloc[:,0].apply(lambda x: vader.polarity_scores(x)['compound'])))

In [ ]:
# This is our baseline. Let's see if we can get close to that with some custom models.
# CountVectorizer in sklearn will do a lot for us, specifically:
# * Tokenise
# * Convert to lowercase
# * Keep counts of tokens in each row of our dataframe

## Do the following:

* Tokenise the input, convert to lowercase
* Use sklearn.feature_extraction.text.CountVectorizer to get features; connect with a classifier
* Consider one or more of the following
  * Remove stopwords
  * Use a word stems / lemmas to reduce feature space
  * CountVectorizer uses a unigram model. Use a 